In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("Sustainability_Data.csv")

In [3]:
print(df.columns)


Index(['Country', 'Year', 'Carbon_dioxide_emissions',
       'Electricity_production_non_renewable',
       'Electricity production_renewable', 'Micro_air_pollution',
       'Greenhouse_emission'],
      dtype='str')


In [4]:
import pandas as pd

# Columns to check for missingness
value_cols = [
    'Carbon_dioxide_emissions',
    'Electricity_production_non_renewable',
    'Electricity production_renewable',
    'Micro_air_pollution',
    'Greenhouse_emission'
]

# Create working table
country_year_missing = df[['Country', 'Year'] + value_cols].copy()

# Missing count and index
country_year_missing['Missing_Count'] = country_year_missing[value_cols].isna().sum(axis=1)
country_year_missing['Missingness_Index'] = (
    country_year_missing['Missing_Count'] / len(value_cols) * 100
).round(2)

# Missingness category for 5 variables
def categorize_missing(count):
    if count == 0:
        return 'Complete'
    elif count == 1:
        return 'Low Missingness(1 Variable)'
    elif count in [2, 3]:
        return 'Moderate Missingness(2-3 Variables)'
    elif count == 4:
        return 'High Missingness(4 Variables)'
    else:
        return 'Fully Missingness'

country_year_missing['Missingness_Category'] = country_year_missing['Missing_Count'].apply(categorize_missing)

# Create decade-style year groups
country_year_missing['Year_Group'] = (
    (country_year_missing['Year'] // 10) * 10
).astype(int).astype(str) + '-' + (
    ((country_year_missing['Year'] // 10) * 10 + 9)
).astype(int).astype(str)

# Overall missingness category summary
category_summary = (
    country_year_missing['Missingness_Category']
    .value_counts()
    .reindex([
        'Complete',
        'Low Missingness(1 Variable)',
        'Moderate Missingness(2-3 Variables)',
        'High Missingness(4 Variables)',
        'Fully Missingness'
    ])
    .reset_index()
)
category_summary.columns = ['Missingness_Category', 'Number_of_rows']
category_summary['Percentage (%)'] = (
    category_summary['Number_of_rows'] /
    category_summary['Number_of_rows'].sum() * 100
).round(2)

# Average missingness by year group
missing_by_year_group = (
    country_year_missing.groupby('Year_Group')['Missingness_Index']
    .mean()
    .round(2)
    .reset_index(name='Average_Missingness_Index')
)
missing_by_year_group = missing_by_year_group.sort_values(
    by='Year_Group',
    key=lambda x: x.str[:4].astype(int)
)

# Missingness category distribution within each year group
category_by_year_group = (
    country_year_missing.groupby(['Year_Group', 'Missingness_Category'])
    .size()
    .reset_index(name='Count')
)

category_by_year_group['Percentage (%)'] = (
    category_by_year_group.groupby('Year_Group')['Count']
    .transform(lambda x: (x / x.sum() * 100).round(2))
)

category_by_year_group = category_by_year_group.sort_values(
    by='Year_Group',
    key=lambda x: x.str[:4].astype(int)
)

# Print outputs
print("\n=== Overall Missingness Category Summary ===")
print(category_summary.to_string(index=False))

print("\n=== Average Missingness by Year Group ===")
print(missing_by_year_group.to_string(index=False))

print("\n=== Missingness Category by Year Group ===")
print(category_by_year_group.to_string(index=False))


=== Overall Missingness Category Summary ===
               Missingness_Category  Number_of_rows  Percentage (%)
                           Complete            3377           23.93
        Low Missingness(1 Variable)            1150            8.15
Moderate Missingness(2-3 Variables)            6686           47.38
      High Missingness(4 Variables)             262            1.86
                  Fully Missingness            2636           18.68

=== Average Missingness by Year Group ===
Year_Group  Average_Missingness_Index
 1970-1979                      76.74
 1980-1989                      76.48
 1990-1999                      24.16
 2000-2009                      19.77
 2010-2019                      30.77
 2020-2029                      68.20

=== Missingness Category by Year Group ===
Year_Group                Missingness_Category  Count  Percentage (%)
 1970-1979                   Fully Missingness   1001           41.60
 1970-1979       High Missingness(4 Variables)     12